In [ ]:
import os
from google.colab import files
import shutil

for subset in ["FD002", "FD003", "FD004"]:
    os.makedirs(f"/content/{subset}", exist_ok=True)
    os.makedirs(f"/content/checkpoints_cgan_{subset}", exist_ok=True)
os.makedirs("/content/figures", exist_ok=True)

print("Upload these 6 files:")
print("  X_train_FD002.npy, labels_train_FD002.npy")
print("  X_train_FD003.npy, labels_train_FD003.npy")
print("  X_train_FD004.npy, labels_train_FD004.npy")
uploaded = files.upload()


In [ ]:
for subset in ["FD002", "FD003", "FD004"]:
    for fname in ["X_train", "labels_train"]:
        src = f"/content/{fname}_{subset}.npy"
        dst = f"/content/{subset}/{fname}.npy"
        if os.path.exists(src):
            shutil.move(src, dst)
            print(f"Moved {src} -> {dst}")

for subset in ["FD002", "FD003", "FD004"]:
    print(f"{subset}: {os.listdir(f'/content/{subset}')}")


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import matplotlib.pyplot as plt
import json
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 4
SEQ_LEN     = 30
INPUT_DIM   = 16
HIDDEN_DIM  = 128
LATENT_DIM  = 64
EMBED_DIM   = 16
BATCH_SIZE  = 64

print(f"Device    : {DEVICE}")
print(f"INPUT_DIM : {INPUT_DIM}")
print(f"Seed      : {SEED}")


In [ ]:
class CMAPSSDataset(Dataset):
    def __init__(self, X, labels):
        self.X      = torch.tensor(X,      dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.labels[idx]


class CGANGenerator(nn.Module):
    def __init__(self, latent_dim, hidden_dim, seq_len,
                 output_dim, num_classes, embed_dim):
        super().__init__()
        self.seq_len     = seq_len
        self.class_embed = nn.Embedding(num_classes, embed_dim)
        self.fc_in  = nn.Sequential(
            nn.Linear(latent_dim + embed_dim, hidden_dim),
            nn.LeakyReLU(0.2),
        )
        self.gru    = nn.GRU(hidden_dim, hidden_dim, num_layers=2, batch_first=True)
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim // 2, output_dim),
            nn.Sigmoid(),
        )
    def forward(self, z, c):
        emb   = self.class_embed(c)
        inp   = torch.cat([z, emb], dim=-1)
        h     = self.fc_in(inp)
        h_seq = h.unsqueeze(1).repeat(1, self.seq_len, 1)
        h_seq = h_seq + 0.1 * torch.randn_like(h_seq)
        out, _ = self.gru(h_seq)
        return self.fc_out(out)


class CGANDiscriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, embed_dim):
        super().__init__()
        self.class_embed = nn.Embedding(num_classes, embed_dim)
        self.gru    = nn.GRU(input_dim + embed_dim, hidden_dim,
                             num_layers=2, batch_first=True)
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, x, c):
        emb  = self.class_embed(c).unsqueeze(1).repeat(1, x.size(1), 1)
        inp  = torch.cat([x, emb], dim=-1)
        _, h = self.gru(inp)
        return self.fc_out(h[-1])


# ── Training helpers ──────────────────────────────────────────────────────────

def correlation_penalty(x_real: torch.Tensor, x_fake: torch.Tensor) -> torch.Tensor:
    """
    BUG-008 fix: penalise difference between real and synthetic
    pairwise feature covariance matrices so the generator learns
    joint sensor relationships, not just marginal distributions.
    """
    B         = x_real.size(0)
    real_flat = x_real.reshape(B, -1)
    fake_flat = x_fake.reshape(B, -1)
    real_c    = real_flat - real_flat.mean(dim=0, keepdim=True)
    fake_c    = fake_flat - fake_flat.mean(dim=0, keepdim=True)
    corr_real = (real_c.T @ real_c) / max(B - 1, 1)
    corr_fake = (fake_c.T @ fake_c) / max(B - 1, 1)
    return torch.mean((corr_real.detach() - corr_fake) ** 2)


def compute_discriminative_score(G, X_real, y_real, latent_dim, device,
                                  n_samples=500, seed=42):
    """RF accuracy on real vs synthetic. Target: ~0.50 (indistinguishable)."""
    G.eval()
    rng       = np.random.default_rng(seed)
    class_ids = np.unique(y_real)
    synth     = []
    with torch.no_grad():
        for c in class_ids:
            n   = min(n_samples // len(class_ids), int((y_real == c).sum()))
            z   = torch.randn(n, latent_dim).to(device)
            c_t = torch.full((n,), int(c), dtype=torch.long).to(device)
            synth.append(G(z, c_t).cpu().numpy())
    X_synth = np.concatenate(synth, axis=0)

    idx_r   = rng.choice(len(X_real),  min(n_samples, len(X_real)),  replace=False)
    idx_s   = rng.choice(len(X_synth), min(n_samples, len(X_synth)), replace=False)
    X_disc  = np.concatenate([X_real[idx_r].reshape(len(idx_r), -1),
                               X_synth[idx_s].reshape(len(idx_s), -1)], axis=0)
    y_disc  = np.array([0]*len(idx_r) + [1]*len(idx_s))
    X_sc    = StandardScaler().fit_transform(X_disc)
    clf     = RandomForestClassifier(n_estimators=50, random_state=seed, n_jobs=-1)
    scores  = cross_val_score(clf, X_sc, y_disc, cv=3, scoring="accuracy")
    G.train()
    return float(scores.mean())


class EarlyStopping:
    """
    BUG-013 fix: stops training when disc_score stops improving toward 0.50.

    Key parameters
    --------------
    min_epochs    : never evaluate or stop before this epoch (warmup guard)
    eval_interval : how often to run the RF discriminative score
    patience      : consecutive non-improving evals before stopping
    min_delta     : improvement must exceed this to count as real progress
    """
    def __init__(self, G, D, X_real, y_real, latent_dim, device, ckpt_dir,
                 eval_interval=100, patience=5, min_epochs=300, min_delta=0.005):
        self.G             = G
        self.D             = D
        self.X_real        = X_real
        self.y_real        = y_real
        self.latent_dim    = latent_dim
        self.device        = device
        self.ckpt_dir      = Path(ckpt_dir)
        self.eval_interval = eval_interval
        self.patience      = patience
        self.min_epochs    = min_epochs
        self.min_delta     = min_delta
        self.best_dist     = float("inf")
        self.no_improve    = 0
        self.stopped_epoch = None

    def step(self, epoch) -> bool:
        if epoch < self.min_epochs:
            return False
        if epoch % self.eval_interval != 0:
            return False

        score = compute_discriminative_score(
            self.G, self.X_real, self.y_real, self.latent_dim, self.device
        )
        dist = abs(score - 0.5)
        print(f"  [EarlyStopping] epoch {epoch}: disc_score={score:.4f}  "
              f"dist={dist:.4f}  best={self.best_dist:.4f}  "
              f"no_improve={self.no_improve}/{self.patience}")

        if self.best_dist - dist >= self.min_delta:
            self.best_dist  = dist
            self.no_improve = 0
            torch.save(self.G.state_dict(), self.ckpt_dir / "generator_best.pt")
            torch.save(self.D.state_dict(), self.ckpt_dir / "discriminator_best.pt")
            print(f"  [EarlyStopping] New best dist={dist:.4f} -- checkpoint saved.")
        else:
            self.no_improve += 1
            if self.no_improve >= self.patience:
                self.stopped_epoch = epoch
                print(f"  [EarlyStopping] No improvement for {self.patience} "
                      f"evals -- stopping at epoch {epoch}.")
                return True
        return False


print("Models and training helpers defined.")


In [ ]:
def train_cgan(subset, epochs=500, multi_condition=False):
    """
    Train CGAN for a given subset.
    multi_condition=True : FD002/FD004 -- lower D LR, throttled D updates,
                           higher min_epochs warmup (400 vs 300).
    multi_condition=False: FD003 -- standard training.
    """
    print(f"\n{'='*55}")
    mode = "multi-condition" if multi_condition else "standard"
    print(f"Training CGAN for {subset} ({mode})")
    print(f"{'='*55}")

    data_dir = Path(f"/content/{subset}")
    ckpt_dir = Path(f"/content/checkpoints_cgan_{subset}")

    X      = np.load(data_dir / "X_train.npy")
    labels = np.load(data_dir / "labels_train.npy")
    print(f"X shape : {X.shape}")
    print(f"Classes : {Counter(labels.tolist())}")

    dataset    = CMAPSSDataset(X, labels)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                            shuffle=True, drop_last=True)

    G = CGANGenerator(LATENT_DIM, HIDDEN_DIM, SEQ_LEN,
                      INPUT_DIM, NUM_CLASSES, EMBED_DIM).to(DEVICE)
    D = CGANDiscriminator(INPUT_DIM, HIDDEN_DIM,
                          NUM_CLASSES, EMBED_DIM).to(DEVICE)

    lr_d   = 5e-5 if multi_condition else 2e-4
    min_ep = 400  if multi_condition else 300

    opt_G = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr_d,  betas=(0.5, 0.999))
    bce   = nn.BCEWithLogitsLoss()

    stopper = EarlyStopping(
        G=G, D=D, X_real=X, y_real=labels,
        latent_dim=LATENT_DIM, device=DEVICE,
        ckpt_dir=ckpt_dir,
        eval_interval=100,
        patience=5,
        min_epochs=min_ep,
        min_delta=0.005,
    )

    print(f"LR_G=1e-4  LR_D={lr_d}  D_throttle={'yes' if multi_condition else 'no'}  "
          f"min_epochs={min_ep}  patience=5  eval_interval=100")

    history_g, history_d = [], []

    for epoch in range(1, epochs + 1):
        g_epoch, d_epoch = 0.0, 0.0

        for x_real, c_batch in dataloader:
            x_real  = x_real.to(DEVICE)
            c_batch = c_batch.to(DEVICE)
            B       = x_real.size(0)
            z       = torch.randn(B, LATENT_DIM).to(DEVICE)

            # ── Discriminator ──────────────────────────────────────────
            x_fake = G(z, c_batch).detach()
            d_real = D(x_real, c_batch)
            d_fake = D(x_fake, c_batch)
            loss_D = (bce(d_real, torch.ones_like(d_real)) +
                      bce(d_fake, torch.zeros_like(d_fake))) / 2.0

            if multi_condition:
                if loss_D.item() > 0.15:
                    opt_D.zero_grad(); loss_D.backward()
                    torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
                    opt_D.step()
                else:
                    loss_D = loss_D.detach()
            else:
                opt_D.zero_grad(); loss_D.backward()
                torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
                opt_D.step()

            # ── Generator x2 ───────────────────────────────────────────
            for _ in range(2):
                z         = torch.randn(B, LATENT_DIM).to(DEVICE)
                x_fake    = G(z, c_batch)
                d_fake    = D(x_fake, c_batch)
                loss_G    = bce(d_fake, torch.ones_like(d_fake))
                loss_fm   = torch.mean(torch.abs(
                    x_real.mean(dim=(0,1)) - x_fake.mean(dim=(0,1))
                ))
                loss_corr = correlation_penalty(x_real, x_fake)
                # BUG-008: correlation penalty weight=0.5
                loss_G_total = loss_G + 10.0 * loss_fm + 0.5 * loss_corr
                opt_G.zero_grad(); loss_G_total.backward()
                torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
                opt_G.step()

            g_epoch += loss_G_total.item()
            d_epoch += loss_D.item()

        g_avg = g_epoch / len(dataloader)
        d_avg = d_epoch / len(dataloader)
        history_g.append(g_avg)
        history_d.append(d_avg)

        if epoch % 100 == 0:
            print(f"  Epoch {epoch:>4}/{epochs}  G={g_avg:.4f}  D={d_avg:.4f}")

        if stopper.step(epoch):
            print(f"  Early stopping triggered at epoch {epoch}.")
            break

    # save -- prefer best checkpoint saved by EarlyStopping
    torch.save(G.state_dict(), ckpt_dir / "generator.pt")
    torch.save(D.state_dict(), ckpt_dir / "discriminator.pt")
    best_g = ckpt_dir / "generator_best.pt"
    if best_g.exists():
        import shutil as _sh
        _sh.copy(best_g, ckpt_dir / "generator.pt")
        _sh.copy(ckpt_dir / "discriminator_best.pt", ckpt_dir / "discriminator.pt")
        print("  Using best checkpoint saved by EarlyStopping.")

    cfg = {
        "seq_len": SEQ_LEN, "input_dim": INPUT_DIM, "hidden_dim": HIDDEN_DIM,
        "latent_dim": LATENT_DIM, "num_classes": NUM_CLASSES, "embed_dim": EMBED_DIM,
        "model_type": "cgan", "subset": subset, "multi_condition": multi_condition,
    }
    with open(ckpt_dir / "model_config.json", "w") as f:
        json.dump(cfg, f, indent=2)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(history_g, label="Generator")
    ax.plot(history_d, label="Discriminator")
    if multi_condition:
        ax.axhline(0.15, color="red", linestyle="--", alpha=0.5, label="D throttle")
    ax.set_title(f"{subset} CGAN Training Losses ({mode})")
    ax.set_xlabel("Epoch")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"/content/figures/cgan_curves_{subset}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    print(f"  Checkpoints saved -> {ckpt_dir}")
    return G


print("train_cgan function defined.")


In [ ]:
# FD003 - single fault, 1 op condition - standard training
G_FD003 = train_cgan("FD003", epochs=500, multi_condition=False)


In [ ]:
# FD002 - single fault, 6 op conditions - throttled D, lower LR
G_FD002 = train_cgan("FD002", epochs=500, multi_condition=True)


In [ ]:
# FD004 - multi fault, 6 op conditions - throttled D, lower LR
G_FD004 = train_cgan("FD004", epochs=500, multi_condition=True)


In [ ]:
import shutil
from google.colab import files

os.makedirs("/content/all_checkpoints", exist_ok=True)

for subset in ["FD002", "FD003", "FD004"]:
    shutil.copytree(
        f"/content/checkpoints_cgan_{subset}",
        f"/content/all_checkpoints/checkpoints_cgan_{subset}",
        dirs_exist_ok=True
    )

shutil.make_archive("/content/multi_cgan_checkpoints", "zip",
                    "/content/all_checkpoints")
files.download("/content/multi_cgan_checkpoints.zip")

shutil.make_archive("/content/multi_cgan_figures", "zip", "/content/figures")
files.download("/content/multi_cgan_figures.zip")

print("Downloads started.")
